# STEN: Hyperparameter tuning on Mumbai PM10 (30- and 90-day horizons)

**Purpose:** find the best STEN hyperparameters for each horizon with the paper's protocol and **save them**, so the testing notebook can retrain and evaluate with exactly these settings.

**Method:** Pathak & Chakraborty, *Deep Generative Spatiotemporal Engression for Probabilistic Forecasting of Epidemics*, TMLR 06/2026, using the official `stengression` package.
**Data:** daily PM10, 17 Mumbai stations, 2021–2024 ([GraphSVR repository](https://github.com/ctanujit/GraphSVR/tree/main/Data)), used as published: no stations dropped, gap-filled values kept.

## Tuning protocol (paper, App. E.1)

| Item | Setting |
|---|---|
| Horizons | 30 and 90 days (both tuned in one run, one after the other) |
| Data split | test = last H days (**never used here**); validation = the 60 (H = 30) or 120 (H = 90) days before it; training (fit) = everything earlier |
| Scaling | z-score per station, using **training (fit) statistics only** |
| Lookback | 60 days |
| Search | Optuna (TPE sampler), **100 trials**, each trained for **100 epochs** (Adam, energy score with β = 1, M = 2 samples per step, batch size 64) |
| Objective | **CRPS on the validation block**, from 100-sample forecasts made from non-overlapping origins that end at the end of the block |
| Search space | Below |

| Hyperparameter | Meaning | Range |
|---|---|---|
| `emb_dim` | STAR output size D′ | 1–10 |
| `L` | maximum spatial lag | 1, 2, 3 |
| `alpha` | distance decay in $W_{ij} = 1/d_{ij}^{\alpha}$ | 1, 2, 3 |
| `hidden` | LSTM hidden size | 2–100 |
| `layers` | LSTM layers | 1–5 |
| `dropout` | dropout between LSTM layers | 0.0–0.5 |
| `lr` | Adam learning rate | 10⁻⁵ – 10⁻¹ (log scale) |
| `noise_dist` | engression noise | Gaussian or uniform |

## What gets saved (folder `STEN_tuning/`, plus a `.zip`)
* **`best_hyperparameters_STEN.json`**: the best configuration for each horizon, with its validation CRPS, the number of completed trials, the data split, the stations, and a fingerprint of the data file. **This is the file the testing notebook reads.**
* `optuna.db`: every trial (so tuning can resume), and `trials_STEN_H30.csv` / `_H90.csv`.

## Running on Kaggle
1. Settings: **Accelerator → GPU**, **Internet → On**. Without internet, attach the data CSVs and the `stengression` wheel as an Input dataset.
2. **Save Version → "Save & Run All (Commit)"**. Only committed versions keep their files under **Output**.
3. **Session limit:** Kaggle stops a session after 12 hours. The notebook stops tuning when `TIME_BUDGET_HOURS` runs out and saves what it has. To continue, attach this notebook's own output as an **Input** and run again: it picks up the saved `optuna.db` and runs only the remaining trials.

---
# 1. Setup

In [1]:
import subprocess, sys, glob
def pip_install(*args):
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True).returncode == 0
try:
    import stengression, optuna, plotly
    print("packages already available")
except ImportError:
    if pip_install("stengression", "optuna", "plotly"):
        print("installed from PyPI")
    else:
        wheels = glob.glob("/kaggle/input/**/stengression-*.whl", recursive=True) + glob.glob("stengression-*.whl")
        assert wheels, "No internet and no stengression wheel found: turn Internet on, or attach the wheel as a Kaggle input."
        print("installed stengression from", wheels[0], pip_install("--no-index", "--no-deps", wheels[0]))

installed from PyPI


In [2]:
import os, json, time, hashlib, shutil, warnings, platform
from pathlib import Path
from importlib.metadata import version as pkg_version
import numpy as np, pandas as pd
import torch, torch.optim as optim
from torch.utils.data import DataLoader
import optuna
import plotly.graph_objects as go
from stengression import MVEN, STEN, energy_score_loss, SpatioTemporalDataset, prepare_spatial_weights

warnings.filterwarnings("ignore", message=".*dropout option adds dropout after all but last recurrent layer.*")
optuna.logging.set_verbosity(optuna.logging.WARNING)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
if device.type == "cpu": torch.set_num_threads(os.cpu_count())
print("PyTorch", torch.__version__, "| Optuna", optuna.__version__, "| device:", device,
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

PyTorch 2.11.0+cu130 | Optuna 5.1.0 | device: cuda | Tesla T4


## 1.1 Settings
These are the paper's values (the same as in the earlier Mumbai PM10 notebooks); **don't change them** for the main experiment. The only switches are for running on Kaggle (time budget) and for a quick flow check.

In [3]:
MODEL_NAME = "STEN"

# ---- data (as published: no stations dropped, gap-filled values kept, no spike repair) ----
DATA_FILE, STATIONS_FILE = "Mumbai_PM10_Data.csv", "Mumbai_Stations_PM10.csv"
DATA_URL     = "https://raw.githubusercontent.com/ctanujit/GraphSVR/main/Data/" + DATA_FILE
STATIONS_URL = "https://raw.githubusercontent.com/ctanujit/GraphSVR/main/Data/" + STATIONS_FILE
START_DATE   = "2021-01-01"

# ---- protocol (paper) ----
HORIZONS   = [30, 90]
VAL_LENS   = {30: 60, 90: 120}
P_LAG      = 60
BATCH_SIZE = 64
N_TRIALS, TUNE_EPOCHS, USE_PRUNING = 100, 100, False
M_TRAIN, M_VAL = 2, 100
TUNE_SEED  = 0

# ---- running switches ----
TIME_BUDGET_HOURS = 11.0   # stop tuning before Kaggle's 12-hour limit; results so far are saved, re-run to continue
QUICK_RUN = False          # True: 3 trials x 2 epochs, only to check the flow

if QUICK_RUN:
    N_TRIALS, TUNE_EPOCHS = 3, 2
    print("QUICK_RUN is ON: these hyperparameters are NOT meaningful.")

ON_KAGGLE = os.path.isdir("/kaggle/working")
OUT_DIR = (Path("/kaggle/working") if ON_KAGGLE else Path("outputs")) / f"{MODEL_NAME}_tuning"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = OUT_DIR / "optuna.db"
print(f"{MODEL_NAME} | horizons {HORIZONS} | {N_TRIALS} trials x {TUNE_EPOCHS} epochs | output -> {OUT_DIR}")

STEN | horizons [30, 90] | 100 trials x 100 epochs | output -> /kaggle/working/STEN_tuning


---
# 2. Data
The files are searched for in the Kaggle inputs, then the current folder, and otherwise downloaded from GitHub. The CSV has no dates: its 1461 rows are the days 2021-01-01 to 2024-12-31, and the 17 columns (`V4 … V30`) are matched to the station list **by order**.

A short fingerprint (MD5) of the data file is stored with the results, so the testing notebook can check it uses exactly the same data.

In [4]:
def find_file(name, url):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if hits: return hits[0]
    if os.path.exists(name): return name
    pd.read_csv(url).to_csv(name, index=False); return name

data_path, stations_path = find_file(DATA_FILE, DATA_URL), find_file(STATIONS_FILE, STATIONS_URL)
raw = pd.read_csv(data_path)
stations = pd.read_csv(stations_path).iloc[:, :3].dropna().reset_index(drop=True)
stations.columns = ["Station", "LAT", "LONG"]
assert raw.shape[1] == len(stations), f"{raw.shape[1]} data columns but {len(stations)} stations"
data_md5 = hashlib.md5(raw.to_csv(index=False).encode()).hexdigest()
raw.index = pd.date_range(START_DATE, periods=len(raw), freq="D", name="Date")
df = raw.rename(columns=dict(zip(raw.columns, stations["Station"])))
station_names = df.columns.tolist(); T, N = df.shape
print(f"data: {data_path} | {T} days ({df.index[0].date()} -> {df.index[-1].date()}) x {N} stations | md5 {data_md5[:12]}")
pd.DataFrame({"column": raw.columns, "station": station_names}).T

data: Mumbai_PM10_Data.csv | 1461 days (2021-01-01 -> 2024-12-31) x 17 stations | md5 676a4cb67289


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
column,V4,V5,V7,V9,V10,V11,V13,V16,V17,V18,V19,V21,V22,V23,V27,V29,V30
station,"Boriwali East,IITM","Boriwali East,MPCB",Chakala-Andheri East,"Chhatrapati International Airport,Terminal 2",Colaba,"Deonar,IITM","Kandiwali East,MPCB",Khinipada-Bhandup West,Kurla,Malad West,Mazgaon,Mulund West,"Navy Nagar,Colaba",Powai,Sion,Vile Parle West,"Worli,MPCB"


In [5]:
def haversine_matrix(lat, lon, R=6371.0):
    lat, lon = np.radians(lat)[:, None], np.radians(lon)[:, None]
    a = np.sin((lat - lat.T) / 2) ** 2 + np.cos(lat) * np.cos(lat.T) * np.sin((lon - lon.T) / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))
dist = haversine_matrix(stations["LAT"].values, stations["LONG"].values)        # (N, N) km, Eq. 1 of the paper
iu = np.triu_indices(N, 1)
print(f"distances between stations: {dist[iu].min():.1f}-{dist[iu].max():.1f} km (median {np.median(dist[iu]):.1f})")

distances between stations: 1.0-37.7 km (median 13.1)


## 2.1 Split and scaling for each horizon
For each horizon we build: the **training (fit) windows** (60-day input → H-day target), the per-station **fit statistics**, and the **validation forecast origins**. The test period is cut off and never touched in this notebook.

In [6]:
values = torch.tensor(df.values, dtype=torch.float32).unsqueeze(-1)            # (T, N, 1)
D = values.shape[-1]

def prepare(h):
    val_len = VAL_LENS[h]
    test_start = T - h; val_start = test_start - val_len
    trainval_raw = values[:test_start]                                          # everything before the test period
    fit_mean = trainval_raw[:val_start].mean(0); fit_std = trainval_raw[:val_start].std(0).clamp(min=1e-6)
    series_std = (trainval_raw - fit_mean) / fit_std                            # scaled with fit statistics only
    loader = DataLoader(SpatioTemporalDataset(series_std[:val_start], P_LAG, h), batch_size=BATCH_SIZE, shuffle=False)
    origins = sorted(range(test_start - h, val_start - 1, -h))                  # windows end at the end of validation
    return dict(h=h, val_len=val_len, test_start=test_start, val_start=val_start, trainval_raw=trainval_raw,
                fit_mean=fit_mean, fit_std=fit_std, series_std=series_std, loader=loader, origins=origins)

prep = {h: prepare(h) for h in HORIZONS}
pd.DataFrame([{"horizon": h, "train (fit)": f"{df.index[0].date()} -> {df.index[p['val_start'] - 1].date()}",
               "validation": f"{df.index[p['val_start']].date()} -> {df.index[p['test_start'] - 1].date()}",
               "test (not used here)": f"{df.index[p['test_start']].date()} -> {df.index[-1].date()}",
               "training windows": len(p["loader"].dataset),
               "validation origins": ", ".join(str(df.index[o].date()) for o in p["origins"])} for h, p in prep.items()])

,horizon,train (fit),validation,test (not used here),training windows,validation origins
0,30,2021-01-01 -> 2024-10-02,2024-10-03 -> 2024-12-01,2024-12-02 -> 2024-12-31,1282,"2024-10-03, 2024-11-02"
1,90,2021-01-01 -> 2024-06-04,2024-06-05 -> 2024-10-02,2024-10-03 -> 2024-12-31,1102,2024-07-05


---
# 3. Model, search space and objective

In [7]:
def spatial_weights(alpha, L):
    W = 1.0 / np.where(dist == 0, np.inf, dist) ** alpha                # 1/d^alpha, diagonal 0
    W = W / W.sum(1, keepdims=True)                                     # row-normalised (as in the official code)
    return [w.to(device) for w in prepare_spatial_weights(torch.tensor(W, dtype=torch.float32), L)]

def build_model(hp, h):
    model = STEN(in_feat_dim=D, num_nodes=N, embedding_dim=hp["emb_dim"], max_spatial_lag=hp["L"],
                 lstm_hidden_dim=hp["hidden"], lstm_num_layers=hp["layers"],
                 lstm_dropout=hp["dropout"] if hp["layers"] > 1 else 0.0,
                 p_lag=P_LAG, t_pred=h, noise_dist=hp["noise_dist"]).to(device)
    return model, spatial_weights(hp["alpha"], hp["L"])

def forward(model, x, W_list): return model(x, W_list)

def fit_model(model, loader, optimizer, epochs, W_list):
    model.fit(loader, optimizer, energy_score_loss, W_list, num_epochs=epochs, m_samples=M_TRAIN,
              device=device, monitor=False, visualize=False)

def suggest_hparams(trial):
    return dict(emb_dim=trial.suggest_int("emb_dim", 1, 10), L=trial.suggest_int("L", 1, 3),
                alpha=trial.suggest_int("alpha", 1, 3), hidden=trial.suggest_int("hidden", 2, 100),
                layers=trial.suggest_int("layers", 1, 5), dropout=trial.suggest_float("dropout", 0.0, 0.5),
                lr=trial.suggest_float("lr", 1e-5, 1e-1, log=True),
                noise_dist=trial.suggest_categorical("noise_dist", ["gaussian", "uniform"]))

**Validation CRPS** (the objective): for each validation origin, 100 forecast samples are drawn and compared with the observed validation values:
$\text{CRPS} = \overline{|X - y|} - \tfrac12\,\overline{|X - X'|}$, averaged over days, stations and origins, in µg/m³ (lower is better). The 100 samples are drawn in one batched forward pass; each copy gets its own noise, which is equivalent to the package's `predict()` loop, only faster.

In [8]:
@torch.no_grad()
def sample_ensemble(model, history, W_list, m, mean_, std_):
    model.eval()
    out = forward(model, history.unsqueeze(0).repeat(m, 1, 1, 1).to(device), W_list)
    return out.cpu() * std_ + mean_                                     # (m, h, N, 1) in µg/m³

def crps_ensemble(ens, y):
    return ((ens - y).abs().mean(0) - 0.5 * (ens.unsqueeze(0) - ens.unsqueeze(1)).abs().mean((0, 1))).mean().item()

def validation_crps(model, W_list, p):
    scores = []
    for o in p["origins"]:
        ens = sample_ensemble(model, p["series_std"][o - P_LAG:o], W_list, M_VAL, p["fit_mean"], p["fit_std"])
        scores.append(crps_ensemble(ens, p["trainval_raw"][o:o + p["h"]]))
    return float(np.mean(scores))

def make_objective(p):
    def objective(trial):
        hp = suggest_hparams(trial)
        model, W_list = build_model(hp, p["h"])
        torch.manual_seed(TUNE_SEED)                                    # same training noise in every trial
        optimizer = optim.Adam(model.parameters(), lr=hp["lr"])
        fit_model(model, p["loader"], optimizer, TUNE_EPOCHS, W_list)
        score = validation_crps(model, W_list, p)
        return score if np.isfinite(score) else 1e10                    # a diverged run scores badly
    return objective

---
# 4. Tuning
For each horizon, an Optuna study stored in `optuna.db`. If a previous run's `optuna.db` is attached as a Kaggle input, it is copied in first, so **finished trials are kept** and only the remaining ones run. Each horizon gets an equal share of the remaining time budget.

In [9]:
if not DB_PATH.exists():
    previous = [p for p in glob.glob(f"/kaggle/input/**/{MODEL_NAME}_tuning/optuna.db", recursive=True)]
    if previous:
        shutil.copy(previous[0], DB_PATH); print("resuming from", previous[0])
storage = f"sqlite:///{DB_PATH}"
study_name = lambda h: f"{MODEL_NAME}_H{h}_{N_TRIALS}x{TUNE_EPOCHS}" + ("_quick" if QUICK_RUN else "")

t_start = time.time(); studies, tune_minutes = {}, {}
for i, h in enumerate(HORIZONS):
    study = optuna.create_study(direction="minimize", study_name=study_name(h), storage=storage, load_if_exists=True,
                                sampler=optuna.samplers.TPESampler(seed=TUNE_SEED))
    done = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
    remaining = max(0, N_TRIALS - done)
    left_sec = TIME_BUDGET_HOURS * 3600 - (time.time() - t_start)
    timeout = max(60, left_sec / (len(HORIZONS) - i))                   # share the remaining budget
    print(f"H={h}: {done} trials done, running up to {remaining} more (time limit {timeout / 3600:.1f} h) ...")
    t0 = time.time()
    if remaining:
        study.optimize(make_objective(prep[h]), n_trials=remaining, timeout=timeout, show_progress_bar=True)
    tune_minutes[h] = (time.time() - t0) / 60
    studies[h] = study
    n_done = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
    print(f"H={h}: {n_done}/{N_TRIALS} trials complete | best validation CRPS {study.best_value:.3f} | "
          f"{tune_minutes[h]:.1f} min this run\n   best: {study.best_params}")

H=30: 0 trials done, running up to 100 more (time limit 5.5 h) ...


  0%|          | 0/100 [00:00<?, ?it/s]

H=30: 100/100 trials complete | best validation CRPS 18.690 | 89.9 min this run
   best: {'emb_dim': 4, 'L': 3, 'alpha': 3, 'hidden': 33, 'layers': 1, 'dropout': 0.21067277031158754, 'lr': 0.0007118303959864516, 'noise_dist': 'gaussian'}
H=90: 0 trials done, running up to 100 more (time limit 9.5 h) ...


  0%|          | 0/100 [00:00<?, ?it/s]

H=90: 100/100 trials complete | best validation CRPS 16.370 | 90.6 min this run
   best: {'emb_dim': 8, 'L': 1, 'alpha': 1, 'hidden': 74, 'layers': 2, 'dropout': 0.2822345609780339, 'lr': 0.00013441469624308442, 'noise_dist': 'gaussian'}


---
# 5. Save the best hyperparameters
`best_hyperparameters_STEN.json` holds one entry per horizon. `complete` is `true` only when all `N_TRIALS` trials have finished. If it's `false`, re-run with this output attached as an input to continue.

In [10]:
def safe_version(p):
    try: return pkg_version(p)
    except Exception: return "n/a"

best = {"model": MODEL_NAME, "dataset": "Mumbai PM10", "data_file": DATA_FILE, "data_md5": data_md5,
        "stations": station_names, "dropped_stations": [], "gap_filled_values": "kept as published", "treat_spikes": False,
        "p_lag": P_LAG, "batch_size": BATCH_SIZE, "m_train": M_TRAIN, "m_val": M_VAL, "tune_seed": TUNE_SEED,
        "n_trials_target": N_TRIALS, "tune_epochs": TUNE_EPOCHS, "pruning": USE_PRUNING, "quick_run": QUICK_RUN,
        "objective": "validation CRPS (100 samples)", "versions": {p: safe_version(p) for p in ["stengression", "torch", "optuna"]},
        "device": str(device), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "python": platform.python_version(), "horizons": {}}
for h, study in studies.items():
    trials = study.trials_dataframe(attrs=("number", "value", "params", "state", "duration"))
    trials.columns = [c.replace("params_", "") for c in trials.columns]
    trials.to_csv(OUT_DIR / f"trials_{MODEL_NAME}_H{h}.csv", index=False)
    n_done = int((trials.state == "COMPLETE").sum())
    p = prep[h]
    best["horizons"][str(h)] = {
        "params": study.best_params, "best_val_crps": study.best_value, "best_trial": study.best_trial.number,
        "trials_complete": n_done, "complete": n_done >= N_TRIALS, "tuning_minutes_last_run": round(tune_minutes[h], 1),
        "val_len": p["val_len"], "validation_period": [str(df.index[p["val_start"]].date()), str(df.index[p["test_start"] - 1].date())],
        "test_period": [str(df.index[p["test_start"]].date()), str(df.index[-1].date())],
        "validation_origins": [str(df.index[o].date()) for o in p["origins"]]}
json.dump(best, open(OUT_DIR / f"best_hyperparameters_{MODEL_NAME}.json", "w"), indent=2)

summary = pd.DataFrame({f"H = {h}": {**v["params"], "validation CRPS": round(v["best_val_crps"], 3),
                                     "trials complete": f"{v['trials_complete']}/{N_TRIALS}"} for h, v in best["horizons"].items()})
display(summary)

,H = 30,H = 90
emb_dim,4,8
L,3,1
alpha,3,1
hidden,33,74
layers,1,2
dropout,0.210673,0.282235
lr,0.000712,0.000134
noise_dist,gaussian,gaussian
validation CRPS,18.69,16.37
trials complete,100/100,100/100


## 5.1 Optimisation history
Each dot is one trial's validation CRPS; the line is the best so far.

In [11]:
for h, study in studies.items():
    t = [x for x in study.trials if x.state == optuna.trial.TrialState.COMPLETE]
    vals = np.array([x.value for x in t]); best_so_far = np.minimum.accumulate(vals)
    fig = go.Figure([go.Scatter(x=[x.number for x in t], y=vals, mode="markers", name="trial", opacity=0.6),
                     go.Scatter(x=[x.number for x in t], y=best_so_far, mode="lines", name="best so far")])
    fig.update_layout(title=f"{MODEL_NAME}, H = {h} days: validation CRPS per trial", xaxis_title="trial",
                      yaxis_title="validation CRPS (µg/m³)", height=360)
    fig.update_yaxes(range=[vals.min() * 0.95, np.percentile(vals, 90) * 1.05] if len(vals) > 3 else None)
    fig.show()
    if len(t) >= 5:
        try: optuna.visualization.plot_param_importances(study).update_layout(title=f"H = {h}: hyperparameter importance", height=340).show()
        except Exception as e: print("importance plot skipped:", e)

In [12]:
zip_path = shutil.make_archive(str(OUT_DIR), "zip", root_dir=OUT_DIR)
print("saved:", sorted(p.name for p in OUT_DIR.iterdir())); print("zip:", zip_path)
print(f"total time this run: {(time.time() - t_start) / 3600:.2f} h")

saved: ['best_hyperparameters_STEN.json', 'optuna.db', 'trials_STEN_H30.csv', 'trials_STEN_H90.csv']
zip: /kaggle/working/STEN_tuning.zip
total time this run: 3.01 h


---
**Next:** in the testing notebook, add this notebook's output as an **Input** (Add Input → Your Work). It reads `best_hyperparameters_STEN.json`, retrains each horizon's best configuration on training + validation with 5 seeds, and evaluates on the test period.